In [10]:
import itertools
import networkx as nx

def hamming_distance(a, b):
    return sum(x != y for x, y in zip(a, b))

def partitions_by_positions(nodes, positions):
    """Partition nodes based on their bits at given positions."""
    groups = {}
    for node in nodes:
        key = tuple(node[pos] for pos in positions)
        groups.setdefault(key, []).append(node)
    return list(groups.values())

def special_msts(n, k):
    all_strings = [format(i, f"0{n}b") for i in range(2**n)]
    
    for combo in itertools.combinations(all_strings, k):
        # build graph
        G = nx.Graph()
        G.add_nodes_from(combo)
        for i in range(k):
            for j in range(i+1, k):
                G.add_edge(combo[i], combo[j],
                           weight=hamming_distance(combo[i], combo[j]))
        
        # MST
        T = nx.minimum_spanning_tree(G, weight="weight")
        
        # check edges > 1
        for a, b, data in T.edges(data=True):
            if data["weight"] > 1:
                # XOR a and b
                xor_val = int(a, 2) ^ int(b, 2)
                xor_bits = format(xor_val, f"0{n}b")
                
                # positions where XOR == 0
                zero_positions = [i for i, bit in enumerate(xor_bits) if bit == "0"]
                
                # partition nodes
                parts = partitions_by_positions(combo, zero_positions)
                
                # find partition containing a and b
                for part in parts:
                    if a in part and b in part and len(part) > 2:
                        yield combo, T, (a, b, part)
                        break


In [19]:
def run_bruteforce(n_min=2, n_max=5, k=3, max_print=3):
    for n in range(n_min, n_max+1):
        count = 0
        for combo, T, (a,b,part) in special_msts(n, k):
            count += 1
            if count <= max_print:
                print(f"[n={n}, k={k}]")
                print("  combo:", combo)
                print("  MST edges:", list(T.edges(data=True)))
                print(f"  special edge: {a}-{b}, partition={part}")
        print(f"Total positives for n={n}, k={k}: {count}\n")

# Example run
run_bruteforce(n_min=1, n_max=6, k=3)


Total positives for n=1, k=3: 0

Total positives for n=2, k=3: 0

Total positives for n=3, k=3: 0

Total positives for n=4, k=3: 0

Total positives for n=5, k=3: 0

Total positives for n=6, k=3: 0

